# Uncensored Chatbot on Paperspace

Deploy your own uncensored AI chat bot in minutes.

**Models available:**
- Samantha (best all-around)
- Dolphin (highly capable)
- Wizard-Vicuna (coding focus)

In [21]:
# Install requirements
!pip install torch transformers accelerate

In [10]:
# Choose your model
# Options: 
# - "Samantha" - conversational, helpful
# - "Dolphin" - uncensored, capable
# - "Wizard-Vicuna" - coding focus

MODEL = "TheBloke/Samantha-1.1-Mistral-7B-GGUF"  # Change to uncensored model of choice

# For truly uncensored, use one of these:
# MODEL = "TheBloke/Dolphin-2.2.1-Mistral-7B-GGUF"
# MODEL = "TheBloke/Samantha-1.1-Mistral-7B-GGUF"

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

print("Loading model...")
print(f"GPU Available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForCausalLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,
    device_map="cuda:0"
)
print("Model loaded!")

Loading model...


OSError: TheBloke/Samantha-1.1-Mistral-7B-GGUF is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`

In [ ]:
# Create chat function
def chat(message, system_prompt=None):
    """Simple chat function."""
    if system_prompt is None:
        system_prompt = "You are a helpful AI assistant. Answer questions honestly and directly."
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": message}
    ]
    
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt").to("cuda:0")
    outputs = model.generate(inputs, max_new_tokens=512, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract just the response
    return response.split("assistant\n\n")[-1].strip()

In [27]:
# Test it!
response = chat("Hello! How are you?")
print(response)

NameError: name 'tokenizer' is not defined

## Streamlit Chat Interface

Run this in a separate cell to get a web UI:

In [8]:
# Install streamlit robustly (avoid distutils uninstall issues)
!pip install --upgrade --ignore-installed streamlit blinker pydeck altair

  Using cached streamlit-1.55.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached cachetools-7.0.5-py3-none-any.whl.metadata (5.6 kB)
  Using cached packaging-26.0-py3-none-any.whl.metadata (3.3 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.4 MB/s eta 0:00:00
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached watchdog-6.0.0-py3-none-manylinux2014_x86_64.whl.metadata (44 kB)
  Using cached narwhals-2.18.0-py3-none-any.whl.metadata (14 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 7.8 MB/s eta 0:00:00
Using cached streamlit-1.55.0-py3-none-any.whl (9.1 MB)
Using cached blinker-1.9.0-py3-none-any.

In [ ]:
# Create app.py
app_code = '''
import streamlit as st
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

st.title("🤖 Uncensored Chatbot")

@st.cache_resource
def load_model():
    MODEL = "microsoft/Phi-3-mini-128k-instruct"
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda:0")
    return model, tokenizer

model, tokenizer = load_model()

if "messages" not in st.session_state:
    st.session_state.messages = []

for msg in st.session_state.messages:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input():
    st.chat_message("user").write(prompt)
    st.session_state.messages.append({"role": "user", "content": prompt})
    
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    messages.extend([{"role": m["role"], "content": m["content"]} for m in st.session_state.messages])
    
    inputs = tokenizer.apply_chat_template(messages, tokenize=True, return_tensors="pt").to("cuda:0")
    outputs = model.generate(inputs, max_new_tokens=4096, temperature=0.7, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n\n")[-1]
    
    st.chat_message("assistant").write(response)
    st.session_state.messages.append({"role": "assistant", "content": response})
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print("Created app.py")
print("Run: streamlit run app.py --server.port 8080")

Created app.py
Run: streamlit run app.py --server.port 8080


In [ ]:
# Run the app (execute as a code cell in the notebook)
!streamlit run app.py --server.port 8080

SyntaxError: invalid syntax (1383897815.py, line 1)

## For GGUF Models (better performance)

If you want to run quantized models:

In [11]:
# Install llama.cpp for GGUF models
!pip install llama-cpp-python

# Example with Dolphin
from llama_cpp import Llama

# Download a quantized model
!wget -O model.gguf https://huggingface.co/TheBloke/Dolphin-2.2.1-Mistral-7b-GGUF/resolve/main/dolphin-2.2.1-mistral-7b.Q4_K_M.gguf

# Load with GPU acceleration (n_gpu_layers=-1 offloads all layers to GPU)
llm = Llama(model_path="model.gguf", n_gpu_layers=-1, n_ctx=4096)

output = llm("Write a story about a sexy ladywolf", max_tokens=4096, temperature=0.7)
print(output['choices'][0]['text'])

--2026-03-23 20:09:13--  https://huggingface.co/TheBloke/Dolphin-2.2.1-Mistral-7b-GGUF/resolve/main/dolphin-2.2.1-mistral-7b.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 3.168.73.129, 3.168.73.38, 3.168.73.111, ...
Connecting to huggingface.co (huggingface.co)|3.168.73.129|:443... connected.
HTTP request sent, awaiting response... 307 Temporary Redirect
Location: /TheBloke/dolphin-2.2.1-mistral-7B-GGUF/resolve/main/dolphin-2.2.1-mistral-7b.Q4_K_M.gguf [following]
--2026-03-23 20:09:13--  https://huggingface.co/TheBloke/dolphin-2.2.1-mistral-7B-GGUF/resolve/main/dolphin-2.2.1-mistral-7b.Q4_K_M.gguf
Reusing existing connection to huggingface.co:443.
HTTP request sent, awaiting response... 302 Found
Location: https://cas-bridge.xethub.hf.co/xet-bridge-us/65403e7339948eb634bfcdff/b1d9df1bf080c026a3b1529e0de572a9e71238ecabb58f22f90343488ab8161e?X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Content-Sha256=UNSIGNED-PAYLOAD&X-Amz-Credential=cas%2F20260323%2Fus-east-1%2Fs3%2Faws4_request&X

llama_model_loader: loaded meta data with 20 key-value pairs and 291 tensors from model.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = ehartford_dolphin-2.2.1-mistral-7b
llama_model_loader: - kv   2:                       llama.context_length u32              = 32768
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count 



The ladywolf was the most beautiful creature in the forest. Her fur was as black as the night sky, and her eyes were a deep, fiery red. She had long, slender legs and a thick, bushy tail that she used to swish back and forth as she walked through the woods.

One day, the ladywolf came across a group of hunters who were out looking for game. They had never seen a creature like her before, and they were fascinated by her beauty. The leader of the hunters, a burly man named Thor, couldn't take his eyes off of her. He was so captivated by her that he forgot all about the hunt and just watched her as she moved gracefully through the forest.

The ladywolf knew that she was the most beautiful creature in the world, and she loved the way the hunters admired her. She decided to play a game with them. She would allow them to follow her through the forest, and she would lead them to a place where they could find plenty of game to hunt. In return, she asked for one thing: they would have to prom

## Deployment Steps

1. Create a Paperspace Gradient or Core notebook
2. Copy this notebook content
3. Select a GPU (A100 or RTX)
4. Run cells in order
5. For web access: `streamlit run app.py --server.port 8080`
6. Use Paperspace's "Share" button or tunnel to access externally